# PureGen-EBM vs BrainWash — Step 1 (energy check) and Step 2 (stage-4 sweep)
Purifies the poisoned task's training images with **mid-run Langevin dynamics of the authors' pretrained EBM**
(`SunayBhat1/puregen-ebm-cinic10-imagenet`, trained on generic ImageNet images from CINIC-10 — no victim data), then
resumes continual learning exactly like BrainWash stage 4. The model class comes from the authors' repo; the Langevin
loop is `puregen_ebm/ebm.py` (paper Eq. 9, authors' constants: eps 1.25e-2, temperature 1e-4).

**Decision rules (fixed before running):**
* Step 1: AUROC(poisoned energy > clean energy) ≥ 0.75 → go; 0.6–0.75 → run a reduced sweep; < 0.6 → the EBM barely sees the noise, stop.
* Step 2 operating point: the step count with the best BWT recovery whose **clean+EBM last-task accuracy** is within 3 points of the undefended clean run (single-seed stand-in for the proposal's "within one seed-std").

Time on an A100: step 1 ≈ 5–10 min; each stage-4 run ≈ 10–30 min (step 2 below is ~10 runs — trim `STEPS` if needed).

**Before you start:** on the cluster, pack the noise you already trained and put it on Google Drive:
```bash
cd ~/Brainwash && tar czf bw_noise_ewc.tgz -C repro/sweep/noise cifar100_ewc_cautious_eps0.1 cifar100_ewc_cautious_eps0.3
```
Download `bw_noise_ewc.tgz`, upload it to `MyDrive/`, then run the next cell once to unpack it to `MyDrive/bw_noise/`.
(The noise pkl already contains the victim checkpoint, so nothing else from the cluster is needed.)

In [ ]:
import os, sys, json, glob
from google.colab import drive; drive.mount('/content/drive')
REPO = '/content/brainwash-repro'
if not os.path.exists(REPO):
    !git clone -b claude/adoring-clarke-97rxkd https://github.com/anushkathagle/brainwash-repro.git $REPO
else:
    !git -C $REPO pull --ff-only
%cd $REPO
!pip -q install huggingface_hub safetensors
import torch
assert torch.cuda.is_available(), 'Runtime -> Change runtime type -> GPU (A100/L4 strongly preferred)'
print('GPU:', torch.cuda.get_device_name(0), '| torch', torch.__version__)
sys.path.insert(0, f'{REPO}/repro/defense')
import colab_utils as cu

In [ ]:
if os.path.exists('/content/drive/MyDrive/bw_noise_ewc.tgz') and not os.path.exists('/content/drive/MyDrive/bw_noise'):
    os.makedirs('/content/drive/MyDrive/bw_noise')
    !tar xzf /content/drive/MyDrive/bw_noise_ewc.tgz -C /content/drive/MyDrive/bw_noise
!ls -la /content/drive/MyDrive/bw_noise/*

In [ ]:
PUREGEN = '/content/PureGen_PoisonDefense'
if not os.path.exists(PUREGEN):
    !git clone --depth 1 https://github.com/SunayBhat1/PureGen_PoisonDefense $PUREGEN
os.environ['PUREGEN_REPO'] = PUREGEN

# ---- where your BrainWash noise lives (copy repro/sweep/noise/<tag>/ folders from the cluster to Drive) ----
NOISE_ROOT = '/content/drive/MyDrive/bw_noise'            # contains cifar100_ewc_cautious_eps0.1/<file>.pkl etc.
RESULTS    = '/content/drive/MyDrive/bw_defense_results'  # shared by both notebooks -> reference runs happen once
METHOD     = 'ewc'
TAG_01     = f'cifar100_{METHOD}_cautious_eps0.1'
TAG_03     = f'cifar100_{METHOD}_cautious_eps0.3'
PKL = {t: cu.find_pkl(NOISE_ROOT, t) for t in (TAG_01, TAG_03)}
print(json.dumps(PKL, indent=1))

STEPS = [50, 100, 150, 300]          # Langevin steps to sweep (authors' default: 150)
from puregen_ebm import load_ebm
_ = load_ebm(device='cuda'); print('pretrained EBM downloaded and loads OK')

## Step 1 — can the EBM see BrainWash noise? (minutes, no training)

In [ ]:
out = f'{RESULTS}/ebm_energy_check'
cu.sh(f'python -m puregen_ebm.energy_check --experiment split_cifar100 --noise_pkl "{PKL[TAG_01]}" "{PKL[TAG_03]}" '
      f'--steps {" ".join(map(str, STEPS))} --out {out}')
from IPython.display import Image, display
for j in sorted(glob.glob(f'{out}/*.json')):
    r = json.load(open(j))
    print(f"\n{os.path.basename(j)}: AUROC poisoned-vs-clean = {r['auroc_poisoned_vs_clean']:.3f} "
          f"(uniform control {r['auroc_uniform_vs_clean']:.3f}) -> {r['verdict']}")
    for p in r['purify']:
        print(f"   steps {p['steps']:4d}: survival {p['survival_l2']:.3f} | PSNR clean {p['psnr_clean']:.1f} dB | "
              f"PSNR purified-poison vs clean {p['psnr_poisoned']:.1f} dB (input {p['input_psnr_poisoned']:.1f})")
    display(Image(j.replace('.json', '_energy.png')))

**Reading step 1:** you want high AUROC, and for some step count: survival well below 1 while PSNR-clean stays high
(roughly ≥ 28 dB). If the poisoned-image PSNR after purification rises above the input PSNR, the EBM is moving poisoned
images back toward their clean versions. If the rule says stop, stop here — that is a reportable result.

## Step 2a — undefended references in this environment

In [ ]:
# undefended references IN THIS ENVIRONMENT (GPU/torch differ from the cluster; ~10-30 min each; skipped if done)
ref_clean = cu.stage4(f'{METHOD}_clean__undefended',        PKL[TAG_01], 'clean', METHOD, RESULTS)
ref_01    = cu.stage4(f'{METHOD}_cautious0.1__undefended',  PKL[TAG_01], 'ours',  METHOD, RESULTS)
cu.table([ref_clean, ref_01])
print('cluster reference (RESULTS_cifar100.md, EWC): clean -5.2 / 68.4 ; cautious eps0.1 -9.5 / 56.4 ; cautious eps0.3 -24.7 / 43.2')

## Step 2b — sweep Langevin steps: utility (clean+EBM) and defense (cautious ε=0.1 + EBM)

In [ ]:
sweep = {}
for s in STEPS:
    d = f'--ebm_purify --ebm_steps {s} --puregen_repo {PUREGEN}'
    sweep[s] = (cu.stage4(f'{METHOD}_clean__ebm{s}',       PKL[TAG_01], 'clean', METHOD, RESULTS, d),
                cu.stage4(f'{METHOD}_cautious0.1__ebm{s}', PKL[TAG_01], 'ours',  METHOD, RESULTS, d))
cu.table([ref_clean, ref_01] + [r for pair in sweep.values() for r in pair], undef=ref_01, clean=ref_clean)

In [ ]:
ok = {s: cu.recovery(p[1]['bwt'], ref_01['bwt'], ref_clean['bwt']) for s, p in sweep.items()
      if ref_clean['acc'] - p[0]['acc'] <= 3}
BEST = max(ok, key=ok.get) if ok else None
print('steps within the utility margin:', {s: f'{v:.1f}%' for s, v in ok.items()})
print('operating point:', BEST, '' if BEST else '-> no step count keeps clean accuracy within 3 points; report the trade-off curve')

## Step 2c — operating point at ε=0.3 (the strong attack)

In [ ]:
assert BEST, 'no operating point selected above'
d = f'--ebm_purify --ebm_steps {BEST} --puregen_repo {PUREGEN}'
ref_03 = cu.stage4(f'{METHOD}_cautious0.3__undefended', PKL[TAG_03], 'ours', METHOD, RESULTS)
def_03 = cu.stage4(f'{METHOD}_cautious0.3__ebm{BEST}', PKL[TAG_03], 'ours', METHOD, RESULTS, d)
print('eps 0.1:'); cu.table([ref_01, sweep[BEST][1]], undef=ref_01, clean=ref_clean)
print('eps 0.3:'); cu.table([ref_03, def_03], undef=ref_03, clean=ref_clean)

**Next (optional):** repeat 2a–2c with `METHOD = 'rwalk'` / `'afec_ewc'` (copy their noise folders too), then 2 more seeds at the
operating point (`seed=` argument of `cu.stage4`). Every log is in `RESULTS` on Drive; `cu.table` re-reads them instantly.